#**Dataset**

**Características da base**

* **Esparsidade alta:** como cada usuário avalia apenas 3–12 dos 20 livros, a matriz terá em torno de 75–80% de valores NaN, o que é realista para sistemas de recomendação.

* **Coerência de gosto:** usuários que gostam de um gênero tendem a dar notas maiores (4–5) para livros desse gênero, permitindo que algoritmos como filtragem colaborativa e SVD encontrem padrões.

* **Ruído controlado:**  livros fora do perfil recebem notas mais baixas (1–4), simulando avaliações variadas.

In [2]:
import pandas as pd
import numpy as np

# ============================================================
# 1. DEFINIÇÃO DOS LIVROS
# ============================================================
livros = [
    'Duna',                              # Ficção Científica
    'Fundação',                          # Ficção Científica
    'Neuromancer',                       # Ficção Científica
    'O Problema dos Três Corpos',        # Ficção Científica
    'Orgulho e Preconceito',             # Romance
    'Jane Eyre',                         # Romance
    'O Morro dos Ventos Uivantes',       # Romance
    'Cem Anos de Solidão',               # Realismo Mágico
    'O Amor nos Tempos do Cólera',       # Realismo Mágico
    'Crime e Castigo',                   # Drama
    'Guerra e Paz',                      # Drama
    'Anna Karenina',                     # Drama
    'O Hobbit',                          # Fantasia
    'O Senhor dos Anéis',                # Fantasia
    'As Crônicas de Nárnia',             # Fantasia
    'O Nome do Vento',                   # Fantasia
    'O Iluminado',                       # Terror
    'Drácula',                           # Terror
    'Frankenstein',                      # Terror
    'It: A Coisa'                        # Terror
]

# Perfis de gosto por gênero (para gerar ratings coerentes)
perfis_genero = {
    'Ficção Científica': ['Duna', 'Fundação', 'Neuromancer', 'O Problema dos Três Corpos'],
    'Romance':           ['Orgulho e Preconceito', 'Jane Eyre', 'O Morro dos Ventos Uivantes'],
    'Realismo Mágico':   ['Cem Anos de Solidão', 'O Amor nos Tempos do Cólera'],
    'Drama':             ['Crime e Castigo', 'Guerra e Paz', 'Anna Karenina'],
    'Fantasia':          ['O Hobbit', 'O Senhor dos Anéis', 'As Crônicas de Nárnia', 'O Nome do Vento'],
    'Terror':            ['O Iluminado', 'Drácula', 'Frankenstein', 'It: A Coisa']
}

# ============================================================
# 2. GERAÇÃO DOS RATINGS COM PERFIS DE GOSTO
# ============================================================
np.random.seed(42)

# Cada usuário tem um gênero favorito e um secundário
generos = list(perfis_genero.keys())
usuarios = [f'U{i:04d}' for i in range(1, 2001)]  # U0001 a U2000

# Matriz de ratings (NaN = não avaliado)
matriz_ratings = pd.DataFrame(
    np.nan,
    index=usuarios,
    columns=livros
)

for user_id in usuarios:
    # Sorteia gênero favorito e secundário
    genero_fav = np.random.choice(generos)
    genero_sec = np.random.choice([g for g in generos if g != genero_fav])

    # Sorteia quantos livros vai avaliar (entre 3 e 12)
    n_avaliacoes = np.random.randint(3, 13)

    # Sorteia quais livros avaliar
    livros_avaliados = np.random.choice(livros, size=n_avaliacoes, replace=False)

    for livro in livros_avaliados:
        # Determina se o livro pertence ao gênero favorito/secundário
        if livro in perfis_genero[genero_fav]:
            nota = np.random.choice([4, 5], p=[0.4, 0.6])
        elif livro in perfis_genero[genero_sec]:
            nota = np.random.choice([3, 4, 5], p=[0.3, 0.4, 0.3])
        else:
            nota = np.random.choice([1, 2, 3, 4], p=[0.2, 0.3, 0.3, 0.2])

        matriz_ratings.loc[user_id, livro] = nota


In [3]:
print("Dimensões da matriz:", matriz_ratings.shape)
print("\nPrimeiras linhas:")
print(matriz_ratings.head())

print("\nEstatísticas de avaliações por usuário:")
print(matriz_ratings.notna().sum(axis=1).describe())

print("\nMédia de notas por livro:")
print(matriz_ratings.mean().round(2))

print("\nTotal de avaliações preenchidas:")
print(f"{matriz_ratings.notna().sum().sum()} de {matriz_ratings.size} "
      f"({matriz_ratings.notna().sum().sum()/matriz_ratings.size*100:.1f}% de esparsidade)")

Dimensões da matriz: (2000, 20)

Primeiras linhas:
       Duna  Fundação  Neuromancer  O Problema dos Três Corpos  \
U0001   2.0       2.0          NaN                         NaN   
U0002   1.0       NaN          NaN                         NaN   
U0003   3.0       NaN          2.0                         NaN   
U0004   NaN       4.0          NaN                         5.0   
U0005   NaN       5.0          NaN                         3.0   

       Orgulho e Preconceito  Jane Eyre  O Morro dos Ventos Uivantes  \
U0001                    NaN        3.0                          NaN   
U0002                    NaN        2.0                          NaN   
U0003                    NaN        1.0                          NaN   
U0004                    4.0        4.0                          3.0   
U0005                    NaN        2.0                          NaN   

       Cem Anos de Solidão  O Amor nos Tempos do Cólera  Crime e Castigo  \
U0001                  NaN                 

**(Atividade) Crie um sistema de recomendacao utilizando o FUNKSVD**